# 레슨 03 — HTML 테이블과 리스트 데이터 정리

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/03/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2003%20%E2%80%94%20HTML%20%ED%85%8C%EC%9D%B4%EB%B8%94%EA%B3%BC%20%EB%A6%AC%EC%8A%A4%ED%8A%B8%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EC%A0%95%EB%A6%AC.ipynb)

이 레슨은 웹 페이지에서 가장 자주 만나는 세 가지 반복 구조를 다룬다. 첫째는 행과 열이 분명한 table, 둘째는 화면 카드처럼 생긴 article, 셋째는 단순 목록인 ul/li이다. 학생은 같은 데이터를 눈으로 보는 방식에서 벗어나, 파이썬이 다루기 쉬운 딕셔너리와 리스트로 바꾸는 과정을 연습한다.

수업 데이터는 모두 합성 fixture다. 실제 학생 개인정보나 외부 사이트 데이터를 쓰지 않는다. 반복 요청을 보내지 않아도 구조를 충분히 연습할 수 있도록 class_dashboard.html, feedback_cards.html, todo_list.html, rubric.csv를 사용한다.

## 학습 목표

1. table의 thead, tbody, tr, th, td 역할을 구분한다.
2. 헤더와 셀을 zip으로 묶어 행 단위 딕셔너리를 만든다.
3. 카드형 UI와 리스트형 UI에서 반복 단위를 찾는다.
4. 퍼센트, 횟수, 분 같은 문자열 값을 숫자로 바꾼다.
5. 여러 출처에서 뽑은 값을 하나의 운영 요약으로 저장한다.

---

## 1. 환경 준비와 데이터 위치 확인

코랩에서는 GitHub raw URL에서 fixture를 읽고, 로컬에서는 현재 레슨 폴더의 data 디렉터리를 읽는다. 이 차이를 한 함수로 감추면 같은 노트북을 코랩과 로컬에서 모두 실행할 수 있다. 먼저 DATA_BASE가 어디를 가리키는지 확인한다.


In [ ]:
import os
import re
import csv
import json
from pathlib import Path
from collections import Counter, defaultdict

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/03/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def read_soup(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def percent_to_int(text):
    return clean_int(text)

def safe_text(node, default=''):
    return node.text.strip() if node else default

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


수업에서 가장 먼저 확인할 것은 실행 환경이다. 파일을 못 읽는 오류는 selector 문제가 아니라 경로 문제인 경우가 많다. DATA_BASE 출력이 예상과 다르면 다음 셀로 넘어가기 전에 경로부터 고친다.

---

## 2. table은 헤더와 행을 따로 읽는다

HTML table은 화면에서는 표처럼 보이지만, 코드에서는 thead의 th와 tbody의 tr을 따로 읽어야 한다. class_dashboard.html에는 수업 운영용 학생 현황 표가 들어 있다.


In [ ]:
dashboard_soup = read_soup('class_dashboard.html')
heading = dashboard_soup.select_one('h1').text.strip()
headers = [th.text.strip() for th in dashboard_soup.select('#class-table thead th')]
rows = dashboard_soup.select('#class-table tbody tr')
print(heading)
print(headers)
print('row count:', len(rows))


헤더는 나중에 딕셔너리 key가 된다. key를 직접 적어도 되지만, 실제 사이트에서는 컬럼 순서가 바뀌거나 컬럼이 추가될 수 있으므로 화면의 th를 기준으로 읽는 편이 더 안전하다.

### 운영 메모

- selector는 좁게 잡는다. table 전체가 아니라 #class-table thead th처럼 목적 위치를 드러낸다.
- 행 개수는 항상 먼저 확인한다. 첫 행만 출력하면 일부 누락을 발견하지 못한다.
- 헤더 문자열은 공백을 제거한 뒤 저장한다. 공백이 섞이면 딕셔너리 key가 달라진다.

---

## 3. 한 행을 딕셔너리로 바꾸기

tr 하나에는 td가 여러 개 들어 있다. td 텍스트를 리스트로 만들고, headers와 같은 순서로 묶으면 한 학생의 기록이 딕셔너리가 된다.


In [ ]:
first_row = rows[0]
first_cells = [td.text.strip() for td in first_row.select('td')]
first_record = dict(zip(headers, first_cells))
print(first_record)


이 방식은 단순하지만 강력하다. 표의 column 이름이 명확할 때는 각 td를 인덱스로 하나씩 꺼내는 것보다 실수가 적다. 학생이 만든 답안을 볼 때도 record에 어떤 key가 있는지 먼저 확인하면 다음 단계의 오류를 빠르게 찾을 수 있다.

---

## 4. 전체 행을 리스트로 정리하기

자동화 결과는 보통 한 행에서 끝나지 않는다. 모든 tr을 순회하며 같은 형태의 딕셔너리를 누적한다. 여기서는 progress, submissions, passed, minutes처럼 숫자로 계산할 값도 같이 변환한다.


In [ ]:
records = []
for tr in rows:
    cells = [td.text.strip() for td in tr.select('td')]
    row = dict(zip(headers, cells))
    row['progress_num'] = percent_to_int(row['progress'])
    row['submissions_num'] = clean_int(row['submissions'])
    row['passed_num'] = clean_int(row['passed'])
    row['minutes_num'] = clean_int(row['minutes'])
    records.append(row)

print(records[0])
print('records:', len(records))


문자열 상태 그대로 두면 정렬과 비교가 흔들린다. 예를 들어 '94%'와 '8%'를 문자열로 비교하면 숫자 비교와 다른 결과가 나올 수 있다. 계산에 쓸 값은 별도 필드로 숫자화하는 습관을 들인다.

---

## 5. 상태와 진도 기준으로 필터링하기

운영 대시보드는 전체 목록보다 조건에 맞는 학생을 찾는 데 의미가 있다. active 학생 중 진도율이 80 이상인 학생, watch 상태인 학생, 제출 수가 많은 학생을 따로 뽑아 보자.


In [ ]:
active_high = [row['student'] for row in records if row['status'] == 'active' and row['progress_num'] >= 80]
watch_students = [row['student'] for row in records if row['status'] == 'watch']
heavy_submitters = [row['student'] for row in records if row['submissions_num'] >= 24]
print('active high:', active_high)
print('watch:', watch_students)
print('heavy submitters:', heavy_submitters)


필터 조건은 수업 운영 기준과 연결된다. active는 정상 진행, watch는 관찰 필요로 볼 수 있다. 기준을 코드에 적을 때는 숫자를 왜 그렇게 잡았는지 설명할 수 있어야 한다.

---

## 6. 코스별 요약 만들기

여러 학생을 코스별로 묶으면 수업 운영자가 한눈에 상태를 볼 수 있다. defaultdict를 사용하면 코스별 total, count, watch 값을 쉽게 누적할 수 있다.


In [ ]:
course_summary = defaultdict(lambda: {'count': 0, 'progress_total': 0, 'minutes_total': 0, 'watch': 0})
for row in records:
    bucket = course_summary[row['course']]
    bucket['count'] += 1
    bucket['progress_total'] += row['progress_num']
    bucket['minutes_total'] += row['minutes_num']
    if row['status'] == 'watch':
        bucket['watch'] += 1

summary_rows = []
for course, values in sorted(course_summary.items()):
    summary_rows.append({
        'course': course,
        'students': values['count'],
        'avg_progress': round(values['progress_total'] / values['count'], 1),
        'avg_minutes': round(values['minutes_total'] / values['count'], 1),
        'watch_count': values['watch'],
    })
print(summary_rows)


요약 테이블은 원본 데이터를 줄여서 보여준다. 단순히 줄이는 것이 아니라 수업 판단에 필요한 질문에 답해야 한다. 어떤 코스에 관찰 학생이 많은지, 평균 진도는 어느 정도인지, 학습 시간이 낮은 코스는 없는지 확인한다.

---

## 7. 카드형 UI 읽기

feedback_cards.html은 표가 아니라 article.feedback-card가 반복되는 구조다. 표처럼 헤더가 없기 때문에 카드 하나에서 어떤 selector와 속성을 읽을지 직접 정해야 한다.


In [ ]:
feedback_soup = read_soup('feedback_cards.html')
feedback_cards = feedback_soup.select('article.feedback-card')
feedback_records = []
for card in feedback_cards:
    feedback_records.append({
        'title': card.select_one('.title').text.strip(),
        'summary': card.select_one('.summary').text.strip(),
        'priority': card['data-priority'],
        'index': int(card['data-index']),
    })
print(feedback_records[:3])


카드형 UI에서는 반복 단위를 잘못 잡는 실수가 잦다. section을 반복하면 카드가 하나로 뭉치고, h2만 반복하면 summary와 priority를 잃는다. 반복 단위는 하나의 완성된 데이터 객체를 담는 태그여야 한다.

---

## 8. 우선순위 카드 추출하기

card의 data-priority 속성은 화면에 보이지 않을 수 있지만 자동화에는 유용하다. high, normal, low를 기준으로 개수를 세고, high 카드 제목만 따로 모은다.


In [ ]:
priority_counts = Counter(card['priority'] for card in feedback_records)
high_titles = [card['title'] for card in feedback_records if card['priority'] == 'high']
print(priority_counts)
print(high_titles)


속성 값은 텍스트보다 안정적인 경우가 많다. 다만 사이트가 바뀌면 속성 이름이 바뀔 수 있으므로, selector와 속성 이름을 수업 노트에 같이 남겨 둔다.

---

## 9. 리스트형 UI 읽기

todo_list.html은 li.todo-item이 반복된다. 리스트는 구조가 단순하지만, 상태가 data-status 속성에 들어 있으므로 텍스트만 읽으면 중요한 정보를 놓친다.


In [ ]:
todo_soup = read_soup('todo_list.html')
todo_items = []
for item in todo_soup.select('li.todo-item'):
    todo_items.append({
        'order': int(item['data-order']),
        'status': item['data-status'],
        'text': item.text.strip(),
    })
print(todo_items[:4])
print(Counter(item['status'] for item in todo_items))


리스트 데이터는 간단해 보이지만 순서가 중요한 경우가 많다. data-order를 숫자로 바꾸어 두면 나중에 정렬하거나 누락 번호를 찾을 수 있다.

---

## 10. 평가 기준 CSV 읽기

rubric.csv는 최종 산출물 평가 기준을 담고 있다. HTML만 다루는 것이 아니라 CSV를 함께 읽어야 실제 자동화 흐름에서 입력과 출력 형식을 연결할 수 있다.


In [ ]:
rubrics = list(csv.DictReader(load_text('rubric.csv').splitlines()))
for row in rubrics:
    row['max_score'] = int(row['max_score'])
print(rubrics)
print('total score:', sum(row['max_score'] for row in rubrics))


CSV를 읽을 때 숫자 컬럼도 문자열로 들어온다. max_score를 합산하려면 int로 변환해야 한다. 웹에서 읽은 값과 파일에서 읽은 값 모두 타입 확인이 필요하다.

---

## 11. 운영 요약 저장하기

마지막으로 여러 구조에서 뽑은 데이터를 하나의 요약으로 저장한다. 여기서는 코스별 요약은 CSV로 저장하고, 카드와 할 일 상태는 JSON으로 저장한다.


In [ ]:
with open('lesson03_course_summary.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['course', 'students', 'avg_progress', 'avg_minutes', 'watch_count'])
    writer.writeheader()
    writer.writerows(summary_rows)

operation_summary = {
    'student_rows': len(records),
    'courses': len(summary_rows),
    'high_feedback_titles': high_titles,
    'todo_status': dict(Counter(item['status'] for item in todo_items)),
    'rubric_total': sum(row['max_score'] for row in rubrics),
}
Path('lesson03_operation_summary.json').write_text(json.dumps(operation_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved csv rows:', len(summary_rows))
print(operation_summary)


저장 파일은 다음 자동화 단계의 입력이 된다. 그래서 컬럼 이름과 JSON key를 사람이 이해할 수 있게 정해야 한다. 저장 전에는 개수, 평균, 상태 분포가 원본과 맞는지 간단히 확인한다.

---

## 수업 정리

이번 레슨의 핵심은 화면 모양이 아니라 반복 단위를 찾는 것이다. table은 tr, 카드 UI는 article.feedback-card, 목록은 li.todo-item이 반복 단위였다. 반복 단위를 찾은 뒤에는 텍스트와 속성을 분리해서 읽고, 계산에 쓸 값은 숫자로 변환했다.

실제 사이트로 확장할 때는 다음 기준을 적용한다.

- 수집 대상이 약관과 robots.txt에 어긋나지 않는지 확인한다.
- 개인정보나 로그인 후 데이터는 수업 예제로 사용하지 않는다.
- 요청 간격과 최대 요청 수를 코드에 둔다.
- 저장 파일에는 수집 시각, 출처, 필터 기준을 남긴다.
- selector가 깨졌을 때 빈 결과를 성공으로 처리하지 않는다.

3강 이후부터는 이렇게 정리한 데이터에 오류 검증, 파일 저장, 반복 실행 기준을 더해 운영 자동화 형태로 확장한다.

---

## 12. selector 디버깅 루틴

selector가 틀리면 대부분 NoneType 오류가 나거나 빈 리스트가 나온다. 이때 바로 정답 selector를 외우게 하면 다음 구조에서 다시 막힌다. 수업에서는 아래 순서로 학생이 스스로 좁혀 가게 한다.

1. 파일이 열렸는지 확인한다. load_text 결과의 앞부분을 짧게 출력해 HTML이 실제로 들어왔는지 본다.
2. 가장 넓은 태그부터 확인한다. table, article, li처럼 큰 반복 단위가 몇 개인지 센다.
3. id와 class를 붙여 좁힌다. #class-table tbody tr, article.feedback-card, li.todo-item처럼 목적이 드러나는 selector를 쓴다.
4. 하나의 반복 단위 안에서 하위 selector를 찾는다. card.select_one('.title')처럼 전체 soup가 아니라 현재 card 안에서 찾는다.
5. 최종 결과 개수와 원본 개수를 비교한다. 원본 행 수와 records 길이가 다르면 반복 범위가 잘못된 것이다.


In [ ]:
raw = load_text('class_dashboard.html')
print(raw[:80])
print('table:', len(BeautifulSoup(raw, 'html.parser').select('table')))
print('rows:', len(BeautifulSoup(raw, 'html.parser').select('#class-table tbody tr')))


이 루틴은 실제 사이트에서도 그대로 쓸 수 있다. 단, 실제 사이트에서는 요청 횟수를 줄이기 위해 HTML을 한 번 받아 변수에 저장하고, 그 변수에서 여러 selector를 테스트한다. 같은 URL을 계속 새로 요청하는 방식은 수업에서도 운영에서도 피한다.

---

## 13. 데이터 품질 검토

자동화 결과가 맞는지 보려면 출력값 하나보다 품질 기준을 확인해야 한다. 이번 레슨에서는 다음 네 가지를 본다.

| 기준 | 확인 방법 | 문제가 있을 때 |
|---|---|---|
| 행 개수 | len(records)가 원본 tr 개수와 같은가 | selector가 너무 넓거나 좁다 |
| key 일관성 | 모든 record가 같은 key를 갖는가 | headers와 cells 길이가 다르다 |
| 타입 안정성 | progress_num, minutes_num이 int인가 | 문자열 비교 오류가 생긴다 |
| 저장 가능성 | CSV fieldnames와 row key가 맞는가 | 저장 파일이 비거나 컬럼이 밀린다 |


In [ ]:
print('records:', len(records), 'rows:', len(rows))
print('keys:', sorted(records[0].keys()))
print('progress type:', type(records[0]['progress_num']).__name__)
print('minutes type:', type(records[0]['minutes_num']).__name__)


학생 답안 검수에서는 완성 코드가 정답과 똑같은지보다 위 네 기준을 만족하는지 보는 편이 좋다. 특히 selector는 여러 방식이 가능하다. 결과 개수와 key, 타입, 저장 산출물이 맞으면 인정할 수 있다.

---

## 14. 운영 보고서로 연결하기

수집한 데이터는 보고서 문장으로 바꿔야 의미가 생긴다. 코스별 평균 진도, 관찰 학생 수, 긴급 피드백 수, pending todo 수는 선생님이 바로 판단할 수 있는 지표다.


In [ ]:
report_lines = []
for row in summary_rows:
    report_lines.append(
        f"{row['course']}: 학생 {row['students']}명, 평균 진도 {row['avg_progress']}%, 관찰 {row['watch_count']}명"
    )
report_lines.append(f"긴급 피드백 {len(high_titles)}건, pending todo {Counter(item['status'] for item in todo_items).get('pending', 0)}건")
print('\n'.join(report_lines))


보고 문장을 만들면 학생은 자동화가 왜 필요한지 이해한다. 단순히 HTML을 긁어오는 것이 목표가 아니라, 반복 작업을 줄이고 운영자가 볼 수 있는 정보로 정리하는 것이 목표다.

---

## 15. 실제 사이트 확장 전 점검

이 레슨의 fixture는 안전한 합성 데이터지만, 실제 사이트로 확장할 때는 기준이 더 엄격하다. 로그인 후 화면, 학생 정보, 유료 콘텐츠, 외부 서비스 데이터는 허가 없이 수집하면 안 된다. 또한 같은 페이지를 짧은 간격으로 반복 요청하면 서비스에 부담을 줄 수 있다.

실제 확장 전에는 다음 항목을 문서에 남긴다.

- 수집 목적: 어떤 업무를 줄이기 위한 자동화인가.
- 수집 범위: 어떤 페이지와 어떤 필드만 읽는가.
- 요청 제한: 최대 페이지 수, 요청 간격, 재시도 횟수는 얼마인가.
- 개인정보 여부: 이름, 연락처, 학습 기록이 들어가는가.
- 저장 위치: CSV, JSON, DB 중 어디에 저장하고 누가 접근하는가.

이 기준을 통과하지 못하면 코드를 작성하지 않는다. 학생에게도 이 기준을 반복해서 알려야 웹 자동화가 단순한 크롤링이 아니라 책임 있는 운영 도구라는 점이 잡힌다.



# 레슨 03 — 실습 문제 정답지

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/03/%EB%A0%88%EC%8A%A8%2003%20%E2%80%94%20HTML%20%ED%85%8C%EC%9D%B4%EB%B8%94%EA%B3%BC%20%EB%A6%AC%EC%8A%A4%ED%8A%B8%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EC%A0%95%EB%A6%AC.ipynb)

HTML 테이블과 리스트 데이터 정리 실습 문제의 모범 답안이다. 출력만 맞는지보다 selector 기준, 반복 단위, 타입 변환, 저장 산출물을 함께 확인한다.

## 0. 환경 셀


In [ ]:
import os
import re
import csv
import json
from pathlib import Path
from collections import Counter, defaultdict

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/03/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def read_soup(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def percent_to_int(text):
    return clean_int(text)

def safe_text(node, default=''):
    return node.text.strip() if node else default

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 정답 — 대시보드 제목 읽기


In [ ]:
dashboard_html = load_text('class_dashboard.html')
soup = BeautifulSoup(dashboard_html, 'html.parser')
print(soup.select_one('h1').text.strip())


### 왜 이 코드가 정답인지

load_text는 코랩과 로컬 경로 차이를 처리한다. BeautifulSoup으로 파싱한 뒤 select_one으로 h1 하나를 읽으면 페이지가 정상적으로 로드되었는지 빠르게 확인할 수 있다.

### 채점 포인트

- 파일을 읽은 뒤 selector가 실제 fixture 구조와 대응하는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- id selector를 빼고 th 전체를 읽어 다른 표와 섞는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 2 정답 — 테이블 헤더 추출하기


In [ ]:
headers = [th.text.strip() for th in soup.select('#class-table thead th')]
print(headers)


### 왜 이 코드가 정답인지

thead th만 선택하면 본문 td와 섞이지 않는다. 헤더를 먼저 리스트로 만들면 이후 각 행을 dict(zip(headers, cells)) 형태로 안정적으로 변환할 수 있다.

### 채점 포인트

- 리스트 길이와 첫 값을 함께 출력해 누락을 빠르게 찾는다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- td 순서를 직접 인덱스로만 처리해 컬럼 변경에 약해지는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 3 정답 — 테이블 행 개수 세기


In [ ]:
rows = soup.select('#class-table tbody tr')
print('rows:', len(rows))


### 왜 이 코드가 정답인지

행 개수는 selector가 맞는지 확인하는 기본 검증이다. tbody tr을 선택하면 헤더 행을 제외한 실제 학생 데이터만 얻는다.

### 채점 포인트

- 문자열 숫자를 계산 전에 변환했는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 퍼센트 문자열을 숫자로 바꾸지 않고 비교하는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 4 정답 — 첫 행 딕셔너리 만들기


In [ ]:
cells = [td.text.strip() for td in rows[0].select('td')]
record = dict(zip(headers, cells))
print(record)


### 왜 이 코드가 정답인지

td 값의 순서와 th 헤더의 순서가 같기 때문에 zip으로 묶을 수 있다. 인덱스로 컬럼명을 직접 적는 방식보다 컬럼 변경에 강하다.

### 채점 포인트

- 저장 산출물의 컬럼 이름과 행 수가 요구사항과 맞는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 카드 텍스트만 읽고 data-priority 속성을 놓치는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 5 정답 — 전체 테이블 리스트 만들기


In [ ]:
records = []
for tr in rows:
    cells = [td.text.strip() for td in tr.select('td')]
    records.append(dict(zip(headers, cells)))
print(records[0])
print(len(records))


### 왜 이 코드가 정답인지

모든 행을 같은 형태로 바꾸면 필터링, 정렬, 저장 단계에서 같은 key를 사용할 수 있다. len(records)를 출력해 누락 여부도 함께 확인한다.

### 채점 포인트

- 실제 사이트 확장 시 요청 간격과 개인정보 여부를 설명하게 한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- CSV에 header를 쓰지 않아 결과 파일 의미가 불분명해지는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 6 정답 — 진도율과 통과율 숫자 변환하기


In [ ]:
for row in records:
    row['progress_num'] = clean_int(row['progress'])
    row['passed_num'] = int(row['passed'])
    row['submissions_num'] = int(row['submissions'])
    row['pass_rate'] = round(row['passed_num'] / row['submissions_num'] * 100, 1)
print(records[0]['student'], records[0]['progress_num'], records[0]['pass_rate'])


### 왜 이 코드가 정답인지

progress는 퍼센트 기호가 붙은 문자열이고 passed, submissions도 문자열이다. 숫자로 변환해야 비교와 평균 계산이 정확해진다.

### 채점 포인트

- 파일을 읽은 뒤 selector가 실제 fixture 구조와 대응하는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- id selector를 빼고 th 전체를 읽어 다른 표와 섞는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 7 정답 — 완료 기준 학생 필터링


In [ ]:
ready_students = [row['student'] for row in records if row['status'] == 'active' and row['progress_num'] >= 80]
print(ready_students)


### 왜 이 코드가 정답인지

진도율만 높아도 watch 상태라면 운영상 따로 확인해야 한다. 상태와 숫자 기준을 함께 쓰면 실제 수업 판단에 가까운 필터가 된다.

### 채점 포인트

- 리스트 길이와 첫 값을 함께 출력해 누락을 빠르게 찾는다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- td 순서를 직접 인덱스로만 처리해 컬럼 변경에 약해지는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 8 정답 — 코스별 평균 진도 계산


In [ ]:
summary = {}
for row in records:
    key = row['course']
    summary.setdefault(key, {'total': 0, 'count': 0})
    summary[key]['total'] += row['progress_num']
    summary[key]['count'] += 1
averages = {k: round(v['total'] / v['count'], 1) for k, v in summary.items()}
print(averages)


### 왜 이 코드가 정답인지

코스별 평균을 만들려면 같은 course에 속한 학생의 progress_num을 누적해야 한다. total과 count를 분리하면 평균 계산 과정이 분명해진다.

### 채점 포인트

- 문자열 숫자를 계산 전에 변환했는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 퍼센트 문자열을 숫자로 바꾸지 않고 비교하는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 9 정답 — 코스별 학생 수 세기


In [ ]:
course_counts = Counter(row['course'] for row in records)
print(dict(course_counts))


### 왜 이 코드가 정답인지

개수 세기는 Counter가 가장 간단하다. 코스별 학생 수는 평균과 함께 운영 화면에서 자주 필요한 요약 값이다.

### 채점 포인트

- 저장 산출물의 컬럼 이름과 행 수가 요구사항과 맞는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 카드 텍스트만 읽고 data-priority 속성을 놓치는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 10 정답 — 피드백 카드 읽기


In [ ]:
feedback_soup = BeautifulSoup(load_text('feedback_cards.html'), 'html.parser')
cards = feedback_soup.select('article.feedback-card')
print(len(cards))
print(cards[0].select_one('.title').text.strip())


### 왜 이 코드가 정답인지

카드형 UI는 표 헤더가 없으므로 카드 하나를 반복 단위로 잡아야 한다. article.feedback-card를 선택하면 제목, 요약, priority 속성을 한 번에 다룰 수 있다.

### 채점 포인트

- 실제 사이트 확장 시 요청 간격과 개인정보 여부를 설명하게 한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- CSV에 header를 쓰지 않아 결과 파일 의미가 불분명해지는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 11 정답 — 긴급 피드백만 필터링


In [ ]:
urgent = []
for card in cards:
    if card['data-priority'] == 'high':
        urgent.append(card.select_one('.title').text.strip())
print(urgent)


### 왜 이 코드가 정답인지

data-priority는 사람이 보는 문장보다 자동화 기준으로 쓰기 좋다. high 값만 필터링하면 운영자가 먼저 확인해야 할 카드를 빠르게 분리할 수 있다.

### 채점 포인트

- 파일을 읽은 뒤 selector가 실제 fixture 구조와 대응하는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- id selector를 빼고 th 전체를 읽어 다른 표와 섞는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 12 정답 — todo 상태 세기


In [ ]:
todo_soup = BeautifulSoup(load_text('todo_list.html'), 'html.parser')
counts = {}
for item in todo_soup.select('li.todo-item'):
    status = item['data-status']
    counts[status] = counts.get(status, 0) + 1
print(counts)


### 왜 이 코드가 정답인지

리스트 텍스트에는 할 일 이름만 있고 상태는 속성에 들어 있다. data-status를 읽어야 진행 상태별 요약을 만들 수 있다.

### 채점 포인트

- 리스트 길이와 첫 값을 함께 출력해 누락을 빠르게 찾는다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- td 순서를 직접 인덱스로만 처리해 컬럼 변경에 약해지는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 13 정답 — 루브릭 CSV 점수 합계 계산


In [ ]:
rubrics = list(csv.DictReader(load_text('rubric.csv').splitlines()))
total_score = sum(int(row['max_score']) for row in rubrics)
print([row['criterion'] for row in rubrics])
print(total_score)


### 왜 이 코드가 정답인지

CSV에서 읽은 숫자는 문자열이다. int 변환 후 합산해야 총점이 정확하고, criterion을 출력하면 어떤 기준을 읽었는지도 검증할 수 있다.

### 채점 포인트

- 문자열 숫자를 계산 전에 변환했는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 퍼센트 문자열을 숫자로 바꾸지 않고 비교하는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 14 정답 — 통합 운영 요약 만들기


In [ ]:
operation_summary = {
    'students': len(records),
    'courses': len(course_counts),
    'urgent_feedback': len(urgent),
    'pending_todos': counts.get('pending', 0),
}
print(operation_summary)


### 왜 이 코드가 정답인지

여러 출처에서 뽑은 값을 하나의 딕셔너리로 묶으면 이후 저장과 보고가 쉬워진다. get을 쓰면 pending이 없을 때도 KeyError 없이 0으로 처리할 수 있다.

### 채점 포인트

- 저장 산출물의 컬럼 이름과 행 수가 요구사항과 맞는지 확인한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- 카드 텍스트만 읽고 data-priority 속성을 놓치는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 문제 15 정답 — 코스 요약 CSV 저장하기


In [ ]:
with open('lesson03_course_summary.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['course', 'students', 'avg_progress'])
    writer.writeheader()
    for course, avg in sorted(averages.items()):
        writer.writerow({'course': course, 'students': course_counts[course], 'avg_progress': avg})
print('saved:', 'lesson03_course_summary.csv', len(averages))


### 왜 이 코드가 정답인지

CSV는 헤더가 있어야 사람이 열었을 때 의미를 바로 알 수 있다. averages와 course_counts를 같은 course key로 묶어 저장하면 요약 데이터가 재사용 가능한 산출물이 된다.

### 채점 포인트

- 실제 사이트 확장 시 요청 간격과 개인정보 여부를 설명하게 한다.
- 출력 형태가 문제 요구와 맞는지 확인한다.
- 이전 셀 변수를 재사용하는 문제는 실행 순서도 함께 확인한다.

### 자주 보이는 오답

- CSV에 header를 쓰지 않아 결과 파일 의미가 불분명해지는 경우가 있다.
- selector가 너무 넓어 불필요한 태그가 함께 잡힌다.
- 결과는 비슷하지만 저장 파일이나 요약 key가 문제 요구와 다르다.

---

## 교사용 마무리 점검

15문제 중 12문제 이상 맞으면 기본 통과로 본다. 다만 문제 5, 8, 12, 15는 이후 자동화 레슨의 기반이 되므로 반드시 피드백한다. 학생이 코드만 맞힌 경우에도 반복 단위, 타입 변환, 저장 산출물의 의미를 말로 설명하게 한다.

---

## 교사용 상세 피드백 기준

### table 단계

문제 1~5는 이후 모든 자동화의 기반이다. 학생이 soup.select를 썼는지보다 어떤 반복 단위를 잡았는지 말로 설명할 수 있는지 확인한다. #class-table thead th와 #class-table tbody tr의 차이를 설명하지 못하면 다음 카드형 UI에서도 selector를 무작정 복사할 가능성이 높다.

평가할 때는 headers 길이와 cells 길이가 같은지 함께 묻는다. 둘이 다르면 dict(zip(...))은 오류 없이 일부 값을 버릴 수 있다. 오류가 나지 않는다고 맞는 코드가 아니라는 점을 알려준다.

### 타입 변환 단계

문제 6~9에서는 progress, passed, submissions를 숫자로 바꾸는 이유를 반드시 확인한다. 학생이 출력 결과만 보고 넘어가면 문자열 비교와 숫자 비교 차이를 놓칠 수 있다. 예를 들어 '100%'와 '9%'를 문자열로 비교하면 기대와 다른 결과가 나온다. progress_num처럼 원본 필드를 보존하면서 새 숫자 필드를 만드는 방식을 권장한다.

Counter와 평균 계산은 정답 코드와 조금 달라도 인정할 수 있다. 단, 코스별 학생 수와 평균 진도 계산이 같은 원본 records에서 나온 값이어야 한다. 서로 다른 selector 결과를 섞으면 보고서 숫자가 맞지 않는다.

### card/list 단계

문제 10~12에서는 텍스트와 속성을 구분해야 한다. feedback 카드의 title은 화면 텍스트지만 priority는 data-priority 속성이다. todo 항목도 텍스트만 읽으면 상태 정보를 잃는다. 실제 UI 자동화에서는 화면에 보이지 않는 속성이 더 안정적인 기준이 되는 경우가 많다는 점을 설명한다.

학생이 soup.select('.title')처럼 전체 문서에서 바로 제목을 찾는 방식으로 풀 수도 있다. 다만 카드별 priority와 묶어야 하는 문제에서는 반드시 card 내부에서 select_one을 사용해야 한다. 데이터끼리 짝이 맞는지가 핵심이다.

### 저장 단계

문제 13~15는 운영 자동화의 끝부분이다. 저장 파일이 만들어지는 것뿐 아니라 헤더, 행 수, 컬럼 의미가 맞는지 확인한다. 학생이 writerow를 반복문 밖에 두면 마지막 값만 저장되거나 빈 파일이 만들어질 수 있다. 저장 후에는 파일명을 출력하고 행 수를 함께 출력하게 한다.

CSV는 사람이 열어 확인하기 좋고, JSON은 중첩 요약을 담기 좋다. 이번 문제는 CSV 저장을 요구하지만 최종 미션에서는 JSON을 추가로 저장해도 좋다. 단, 산출물 이름과 내용이 제출 요건과 맞아야 한다.

## 문제별 빠른 확인표

| 문제 | 핵심 검수 | 통과 기준 |
|---:|---|---|
| 1 | 파일 로드와 h1 selector | 제목 한 줄 출력 |
| 2 | thead th 선택 | 헤더 리스트 출력 |
| 3 | tbody tr 선택 | 원본 행 개수 출력 |
| 4 | headers와 cells 매핑 | 첫 행 dict 출력 |
| 5 | 모든 행 반복 | records 길이 확인 |
| 6 | 숫자 필드 추가 | progress_num과 pass_rate 생성 |
| 7 | 복합 조건 필터 | active와 진도 기준 동시 적용 |
| 8 | course별 평균 | total/count 구조 사용 |
| 9 | Counter 사용 | course별 개수 출력 |
| 10 | 카드 반복 단위 | article.feedback-card 선택 |
| 11 | 속성 필터 | data-priority high만 추출 |
| 12 | 리스트 상태 | data-status별 개수 계산 |
| 13 | CSV 숫자 합계 | max_score 합계 100 |
| 14 | 통합 요약 | 여러 출처 값 결합 |
| 15 | CSV 저장 | header와 행 수 출력 |

## 수업 중 멈춤 기준

학생 다수가 문제 4에서 막히면 table 구조 설명으로 돌아간다. 문제 6에서 막히면 타입 변환만 별도 예제로 보여준다. 문제 11에서 막히면 카드 하나를 출력해 title과 data-priority가 같은 article 안에 있음을 확인한다. 문제 15에서 막히면 CSV 저장 형식보다 먼저 저장하려는 rows의 형태를 print로 확인한다.

## 채점 운영 팁

정답 코드는 기준 예시일 뿐이다. 학생 코드가 다른 selector를 사용해도 같은 반복 단위와 같은 결과를 만들면 통과로 볼 수 있다. 예를 들어 '#class-table tbody tr' 대신 'table#class-table tbody tr'을 써도 의미는 같다. 반대로 결과가 우연히 한 번 맞더라도 반복 단위가 너무 넓거나 타입 변환이 빠졌다면 보완을 요구한다.

채점할 때는 다음 순서로 본다.

1. 실행 순서: 환경 셀부터 마지막 셀까지 새 런타임에서 동작하는가.
2. 데이터 개수: 원본 행, 카드, 리스트 항목 개수와 결과 개수가 일치하는가.
3. 데이터 형태: 리스트 안에 같은 key를 가진 딕셔너리가 들어 있는가.
4. 숫자 처리: 평균과 비교에 쓰는 값이 숫자인가.
5. 산출물: CSV 또는 JSON이 비어 있지 않고 헤더가 있는가.

학생이 막혔을 때 바로 답을 알려주기보다 다음 질문을 던진다. 지금 선택한 selector는 몇 개를 반환하는가. 첫 번째 item 안에는 어떤 하위 태그가 있는가. 지금 비교하는 값의 type은 무엇인가. 저장하려는 row를 print하면 어떤 딕셔너리인가. 이 네 질문으로 대부분의 문제를 해결할 수 있다.

## 확장 답안 허용 기준

우수 학생은 함수를 더 많이 분리하거나 JSON 저장을 추가할 수 있다. parse_table_records, summarize_courses, parse_feedback_cards처럼 이름이 분명한 함수로 나누면 가산점을 줄 수 있다. 단, 함수가 많아져도 최종 산출물은 요구한 CSV 또는 JSON 파일로 남아야 한다. 화면 출력만 있고 저장 파일이 없으면 운영 자동화 완성으로 보지 않는다.
## 재실행 검수 기준

교사용 검수에서는 노트북을 새 런타임에서 실행했을 때도 같은 파일이 생성되는지 확인한다. 중간 셀에만 존재하는 임시 변수에 기대어 동작하는 답안은 실제 수업 운영 자동화로 보기 어렵다. 저장 파일을 삭제한 뒤 다시 실행해도 같은 이름과 같은 행 수가 나오면 안정성이 높다.



# 레슨 03 — 최종 미션 모범 답안

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/03/%EB%A0%88%EC%8A%A8%2003%20%E2%80%94%20HTML%20%ED%85%8C%EC%9D%B4%EB%B8%94%EA%B3%BC%20%EB%A6%AC%EC%8A%A4%ED%8A%B8%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%EC%A0%95%EB%A6%AC.ipynb)

대시보드 표, 피드백 카드, todo 리스트, 루브릭 CSV를 읽어 운영 요약 CSV와 JSON을 만든다.

## 0. 환경 셀


In [ ]:
import os
import re
import csv
import json
from pathlib import Path
from collections import Counter, defaultdict

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/03/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def read_soup(filename):
    return BeautifulSoup(load_text(filename), 'html.parser')

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def percent_to_int(text):
    return clean_int(text)

def safe_text(node, default=''):
    return node.text.strip() if node else default

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


## 모범 답안


In [ ]:
dashboard_soup = read_soup('class_dashboard.html')
headers = [th.text.strip() for th in dashboard_soup.select('#class-table thead th')]
records = []
for tr in dashboard_soup.select('#class-table tbody tr'):
    row = dict(zip(headers, [td.text.strip() for td in tr.select('td')]))
    row['progress_num'] = percent_to_int(row['progress'])
    row['minutes_num'] = clean_int(row['minutes'])
    records.append(row)

course_summary = defaultdict(lambda: {'count': 0, 'progress_total': 0, 'minutes_total': 0, 'watch': 0})
for row in records:
    bucket = course_summary[row['course']]
    bucket['count'] += 1
    bucket['progress_total'] += row['progress_num']
    bucket['minutes_total'] += row['minutes_num']
    if row['status'] == 'watch':
        bucket['watch'] += 1

summary_rows = []
for course, values in sorted(course_summary.items()):
    summary_rows.append({
        'course': course,
        'student_count': values['count'],
        'avg_progress': round(values['progress_total'] / values['count'], 1),
        'avg_minutes': round(values['minutes_total'] / values['count'], 1),
        'watch_count': values['watch'],
    })

feedback_soup = read_soup('feedback_cards.html')
high_titles = [card.select_one('.title').text.strip() for card in feedback_soup.select('article.feedback-card') if card['data-priority'] == 'high']

todo_soup = read_soup('todo_list.html')
todo_counts = Counter(item['data-status'] for item in todo_soup.select('li.todo-item'))

rubrics = list(csv.DictReader(load_text('rubric.csv').splitlines()))
rubric_total = sum(int(row['max_score']) for row in rubrics)

with open('lesson03_final_summary.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['course', 'student_count', 'avg_progress', 'avg_minutes', 'watch_count'])
    writer.writeheader()
    writer.writerows(summary_rows)

operation_note = {
    'student_rows': len(records),
    'high_feedback_count': len(high_titles),
    'high_feedback_titles': high_titles,
    'todo_status': dict(todo_counts),
    'rubric_total': rubric_total,
}
Path('lesson03_final_note.json').write_text(json.dumps(operation_note, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', 'lesson03_final_summary.csv', len(summary_rows))
print(operation_note)


## 왜 이 풀이가 기준 답안인지

표, 카드, 리스트, CSV를 각각 다른 방식으로 읽되 최종 결과는 운영자가 볼 수 있는 요약으로 통일했다. 숫자로 계산할 값은 모두 변환했고, 저장 파일에는 코스별 학생 수와 평균 진도, 관찰 학생 수가 들어간다. high priority 피드백과 todo 상태 분포도 JSON에 남겨 다음 보고서 단계에서 바로 재사용할 수 있다.

## 채점 메모

- records 길이가 원본 table 행 수와 같은지 확인한다.
- course_summary의 count 합계가 records 길이와 같은지 확인한다.
- watch_count는 status가 watch인 행만 세야 한다.
- rubric_total은 100이 되어야 한다.
- 실제 사이트로 확장할 때 요청 간격과 개인정보 확인 기준을 설명할 수 있어야 한다.
